# Study 923 — The Cash Lag 💤

**Cash vehicles reprice at different speeds. Can you rotate between them for a yield pickup?**

A bill fund holds a ladder, so its yield is roughly the average of the rates at which its
holdings were bought — it inherits a rate change only as the ladder rolls. A floating-rate
note fund, whose coupon resets weekly, inherits it almost at once. The folk conclusion:
**rates rising → sit in the fast repricer (USFR); rates falling → sit in the slow one
(SHV), which keeps a stale-high yield and books a duration gain on top.**

We test both halves on **BIL, SGOV, USFR, SHV** against **^IRX** (the 13-week bill quote),
2014-02-04 → 2026-06-30 (3,118 days), every arm **excess of BIL's own total
return**, 2 bps one-way.

*Real-tape numbers below are the frozen headline (`docs/results.md`, Fingerprint
`d78f3661ba09`); the live cells run the fast synthetic control and are labelled as such.
As-of 2026-06-30.*


## 1. Why one cash fund is slower than another

Imagine two people buying three-month Treasury bills. One buys a fresh bill every week; after a month, most of what she owns was bought at *today's* rate. The other bought a year's worth at once; she is still earning last year's rate and will be for months. Same asset, same safety — different **speed**.

That speed has a name (weighted average maturity) and a shadow (duration): the slower fund also gets *marked down* when rates jump, because the old, lower-yielding bills it holds are suddenly worth less. So a rate rise hurts the slow fund twice — a mark-down today, and weeks of stale yield afterwards.

> 🔬 **For the quants** — a ladder of *n*-day bills has duration ≈ *n*/2 days, so the ordering below is a *prediction* of bond arithmetic, not a discovery.

## 2. The tape agrees, emphatically

We measured each fund's realised sensitivity to a move in the 13-week bill rate — no clever modelling, just daily returns against daily rate changes:

| Fund | What it holds | Realised duration | HAC *t* |
|---|---|--:|--:|
| **USFR** | floating-rate notes, coupon resets weekly | **-0.001 yr** | -0.04 |
| **SGOV** | 0-3 month bills | **+0.072 yr** | +5.79 |
| **BIL** | 1-3 month bills | **+0.083 yr** | +7.57 |
| **SHV** | 0-1 year Treasuries | **+0.177 yr** | +8.05 |

The order is exactly the maturity order, and every non-zero number is about as certain as anything in finance ever gets. **USFR's is a clean zero** — a bond whose coupon resets every week really does not care where rates go.

In [1]:
R = dict(dur={'USFR': -0.001, 'SGOV': 0.072, 'BIL': 0.083, 'SHV': 0.177}, dur_t={'USFR': -0.04, 'SGOV': 5.79, 'BIL': 7.57, 'SHV': 8.05})
for v in ('USFR','SGOV','BIL','SHV'):
    print('%-5s realised duration %+.3f yr   (HAC t = %+5.2f)'
          % (v, R['dur'][v], R['dur_t'][v]))

USFR  realised duration -0.001 yr   (HAC t = -0.04)
SGOV  realised duration +0.072 yr   (HAC t = +5.79)
BIL   realised duration +0.083 yr   (HAC t = +7.57)
SHV   realised duration +0.177 yr   (HAC t = +8.05)


## 3. So the lag is real. Now: is it worth anything?

The rule writes itself. Rates rising? Hold the fast one, and skip the mark-down. Rates falling? Hold the slow one, keep its stale-high yield and pocket a small gain as its old bills become valuable. We checked the direction of the bill rate over the last month, waited a day (no peeking), and rotated.

**Before any trading costs at all, it earned -6.9 basis points a year** — that is 0.07 of one percent, in the *wrong* direction, with a *t* of -0.29. Zero, in other words. Charge a realistic two basis points a trade and it becomes **-115.3 bp/yr**.

In [2]:
R = dict(sw_gross=-6.9, sw_gross_t=-0.29, sw_net=-115.3, sw_net_t=-4.39,
         static_usfr=16.7, n_switches=333, switches_per_yr=26.9)
print('switch rule, before costs : %+7.1f bp/yr   (t = %+5.2f)'
      % (R['sw_gross'], R['sw_gross_t']))
print('switch rule, after costs  : %+7.1f bp/yr   (t = %+5.2f)'
      % (R['sw_net'], R['sw_net_t']))
print('best fund, just held      : %+7.1f bp/yr' % R['static_usfr'])
print('trades needed             : %d  (%.0f a year)'
      % (R['n_switches'], R['switches_per_yr']))

switch rule, before costs :    -6.9 bp/yr   (t = -0.29)
switch rule, after costs  :  -115.3 bp/yr   (t = -4.39)
best fund, just held      :   +16.7 bp/yr
trades needed             : 333  (27 a year)


## 4. Why it can't work — the prize is smaller than the ticket

Here is the whole problem in two numbers. The *entire* gap between the best and worst cash fund on this tape is about **17 basis points a year**. One round trip — sell one fund, buy another — costs about **4 basis points**. The rule trades **27 times a year**.

You are spending roughly a percent of friction chasing a sixth of a percent of prize. And we know the losses are friction rather than bad luck, because two controls lose in exactly the same way: running the rule **backwards** loses -79 bp/yr, and a **random** switch that trades just as often loses -104 bp/yr. Direction has nothing to do with it.

## 5. The one thing that does work — and it isn't a trade

Since 2020, simply **owning SGOV instead of BIL** — no timing, no rotation, one decision, ever — has beaten BIL by **+12.3 bp/yr** with a *t* of **+3.98**, and it is positive in both halves of that sample (though the recent half is only half as big). Part of it is a cheaper fee, part is the shorter ladder. Buying it costs one round trip — about 4 basis points, paid back in **under four months** — and then nothing, ever again.

One caveat we owe you: we picked SGOV **after** looking at the three candidates, so its *t*-stat is not the reason to believe it. The reason is that the fee gap driving it was published in advance and did not need to be discovered.

> 🔬 **For the quants** — bootstrap CI [+8.0, +16.5] bp/yr, clear of zero; split-half +17.0 (*t* = +3.18) then +7.7 (*t* = +1.96). This is a fee-and-maturity identity, not a risk premium — which is precisely why it survives a hindsight-selection charge that a return anomaly would not.

## 6. Is the machine broken? (live synthetic check — no real data)

Before believing a zero, check that the detector can see anything at all. We build three make-believe worlds and run the *same* code on them. In the middle world, rate moves are made completely unpredictable — and the rotation *still* earns a small, reliable amount, purely because the funds' yields differ. That is the smallest thing this harness needs to notice, and it notices it easily.

**The real tape came in below even that floor.**

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from cash_lag import data, strategy as st

# SYNTHETIC ONLY - three worlds, one detector. No real-tape data is touched here.
worlds = [
    ('real ladder + trending rate ', dict(signal_strength=1.0)),
    ('real ladder + random walk   ', dict(signal_strength=1.0, trend_phi=0.0)),
    ('no ladder (the null)        ', dict(signal_strength=0.0)),
]
for label, kw in worlds:
    d = st.synthetic_detect(*data.synthetic_panel(seed=923, **kw))
    print('%s duration spread %+.3f yr | switch gross %+7.1f bp/yr (t=%+6.2f)'
          % (label, d['duration_spread'], d['gross_bp'], d['gross_t']))

real ladder + trending rate  duration spread +0.222 yr | switch gross  +137.8 bp/yr (t=+16.02)


real ladder + random walk    duration spread +0.210 yr | switch gross    +9.6 bp/yr (t= +5.30)


no ladder (the null)         duration spread +0.000 yr | switch gross    -0.5 bp/yr (t= -0.52)


## Verdict

- **Signal — Mixed.** The lag is real, large and ordered exactly as bond arithmetic predicts (durations -0.001 → +0.177 years, *t* up to 8.1). The *rotation* is not: -6.9 bp/yr before costs, *t* = -0.29. Knowing how a fund lags tells you nothing about which to hold next, because last month's rate move does not forecast next month's.
- **Tradability — Mirage.** Net -115 bp/yr, and a random switch that trades as often loses the same — the loss is friction, full stop. The only bankable finding is a **fund swap, not a trade**: hold SGOV rather than BIL for +12.3 bp/yr and then leave it alone.